# 02: DNA Language Model Perplexity Benchmark on *Anopheles gambiae*

This notebook computes masked language model (MLM) perplexity for three DNA foundation models on real *Anopheles gambiae* reference genome sequences, and ranks them.

**The single-sentence experiment:** for each model, mask 15% of tokens in a batch of mosquito sequences, ask the model to predict the masked tokens, measure how wrong the predictions are, and average the wrongness across the batch. Lower average wrongness indicates a better fit between the model's pretrained knowledge and *An. gambiae* genomic content.

Three foundation models are compared:

| Model | Tokenization | Parameters | Loss type |
|---|---|---|---|
| Nucleotide Transformer v2 | Fixed 6-mer | 500M | Masked LM |
| DNABERT-2 | Byte-pair encoding (BPE) | 117M | Masked LM |
| HyenaDNA (medium) | Single nucleotide | ~7M | Causal LM |

The masked LM (MLM) losses for NT-v2 and DNABERT-2 are directly comparable. HyenaDNA is autoregressive (causal LM), so its loss measures next-token prediction rather than mask-filling. Both quantify "how well does this model predict this DNA," but the numeric scales differ.

**Data source:** *Anopheles gambiae* AgamP4 reference genome, chromosome 3L. Random 500 bp windows are sampled to form the evaluation batch.

**Kernel:** Python 3.11 (Mathieson-LM)

## Setup

Package imports and path configuration. The reference genome FASTA lives on the D: drive rather than in the project folder to conserve limited space on the C: drive.

In [ ]:
# Standard imports
import sys, os, warnings
from pathlib import Path

warnings.filterwarnings('ignore')

# --- Project path resolution ---
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
MODELS_DIR = PROJECT_ROOT / 'models'
OUTPUTS_DIR = PROJECT_ROOT / 'outputs'
OUTPUTS_DIR.mkdir(exist_ok=True)

# --- External data path (kept off C: drive to save space) ---
# AgamP4 reference genome, uncompressed FASTA (~275 MB)
REFERENCE_PATH = Path(r'D:\Mathieson_data\AgamP4.fa')

# --- HuggingFace cache configuration ---
# Redirect model downloads to the project folder (about 5 GB total across three models)
# Note: Preston has considered moving this to D: as well if C: fills further
os.environ['HF_HOME'] = str(MODELS_DIR)

# --- Scientific Python and ML imports ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM, AutoModelForCausalLM

# --- Device selection ---
# GPU dramatically speeds up model inference; the RTX 3060's 12 GB VRAM is sufficient for all three models
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU:    {torch.cuda.get_device_name(0)}')
    print(f'VRAM:   {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# --- Verify the reference genome exists ---
if REFERENCE_PATH.exists():
    size_mb = REFERENCE_PATH.stat().st_size / (1024 * 1024)
    print(f'\nReference genome:  {REFERENCE_PATH}  ({size_mb:.1f} MB)')
else:
    print(f'\nWARNING: Reference genome not found at {REFERENCE_PATH}')
    print('Download from Ensembl Metazoa release-58 and extract to D:\\Mathieson_data\\')

## Loading real mosquito sequences from the AgamP4 reference

Random 500 bp windows are sampled from chromosome 3L of the *An. gambiae* AgamP4 reference genome. Chromosome 3 is used because it lacks the large polymorphic inversions found on chromosome 2 (which would complicate demographic inference in downstream work) and does not have the hemizygous behavior of the X chromosome. Chromosome 3L specifically has ~49 Mb of sequence, providing many candidate windows.

**Filtering:** windows with more than 5% ambiguous bases (`N`) are rejected. Reference assemblies contain gaps in poorly assembled regions, and DNA language models were not trained on such content. Excluding these ensures the perplexity measurement reflects the model's ability to predict biological sequence rather than random symbols.

**Determinism:** the random seed fixes which windows are drawn, so runs are reproducible.

In [ ]:
import pyfaidx

def load_sequences(n=50, length=500, chromosome='3L', n_max_fraction=0.05, seed=42):
    """
    Load n random-window DNA sequences from the AgamP4 reference genome.

    Parameters
    ----------
    n : int
        Number of sequences to return.
    length : int
        Length of each window in base pairs.
    chromosome : str
        FASTA record identifier to sample from (e.g., '3L', '3R', '2L').
    n_max_fraction : float
        Maximum fraction of ambiguous bases ('N') allowed per window.
    seed : int
        RNG seed for reproducibility.

    Returns
    -------
    list of str
        n uppercase DNA sequences of the requested length.
    """
    # pyfaidx.Fasta opens the FASTA lazily and creates a .fai index next to it on first access
    # The index allows random access by chromosome and coordinate without loading the whole file
    fasta = pyfaidx.Fasta(str(REFERENCE_PATH))

    # Get the chromosome as a sliceable record
    chrom = fasta[chromosome]
    chrom_len = len(chrom)

    # Use a NumPy Generator for reproducible random integer sampling
    rng = np.random.default_rng(seed)

    sequences = []
    attempts = 0
    # Cap total attempts at n * 20 in case the chromosome has many N-rich regions
    max_attempts = n * 20

    while len(sequences) < n and attempts < max_attempts:
        # Pick a random start coordinate that leaves room for the full window
        start = int(rng.integers(0, chrom_len - length))
        # Slice the chromosome; pyfaidx returns a Sequence object, str() converts to plain string
        # .upper() normalizes case (some reference sequences use lowercase for soft-masked repeats)
        seq = str(chrom[start:start + length]).upper()

        # Reject windows with too many ambiguous bases (assembly gaps or unresolved regions)
        n_count = seq.count('N')
        if n_count / length <= n_max_fraction:
            sequences.append(seq)
        attempts += 1

    if len(sequences) < n:
        print(f'Warning: only found {len(sequences)}/{n} valid windows after {attempts} attempts')

    return sequences

# --- Load 50 sequences of length 500 bp from chromosome 3L ---
sequences = load_sequences(n=50, length=500, chromosome='3L')

print(f'Loaded {len(sequences)} sequences of length {len(sequences[0])} bp from chromosome 3L')
print(f'First sequence (first 80 bp): {sequences[0][:80]}...')

# Empirical GC content across the batch (An. gambiae is ~44%)
actual_gc = np.mean([(s.count('G') + s.count('C')) / len(s) for s in sequences])
print(f'Empirical GC content:         {actual_gc:.2%}  (expected ~44% for An. gambiae)')

## Masked LM perplexity: how it works

The procedure for each sequence in the evaluation batch:

1. Tokenize the sequence into the model's vocabulary units.
2. Randomly pick 15% of the token positions to mask.
3. Replace those tokens with the model's special `[MASK]` token.
4. Feed the masked sequence to the model.
5. The model outputs, for each position, a probability distribution over the entire vocabulary.
6. Compute cross-entropy loss between the model's predicted distributions and the true (masked-out) tokens, restricted to the masked positions.
7. Average the loss across all masked positions.

The average loss is the model's masked LM loss on that sequence. Perplexity is `exp(loss)`. Both are monotonic: lower loss ↔ lower perplexity ↔ better prediction.

**Reference values:**

- A perfectly random model has loss ≈ `log(vocab_size)`. For a 4,105-token vocabulary that is ≈ 8.3.
- A perfect model that always predicts the masked token has loss = 0.
- Real DNA-LMs on in-distribution data typically land in the range 1 to 4.

**Caveat about HyenaDNA:** HyenaDNA is a causal (autoregressive) language model. It predicts each token from the tokens preceding it, not from bidirectional context around a mask. It has no `[MASK]` token. For HyenaDNA, next-token perplexity is computed instead. Both measure predictive quality, but the numeric scales are not directly comparable because next-token prediction has only ~4 possible outputs (single-nucleotide vocabulary) while masked-LM prediction has thousands.

## Loss computation helpers

Two functions: `compute_mlm_loss` handles BERT-style masked language models (NT-v2 and DNABERT-2). `compute_causal_loss` handles autoregressive models (HyenaDNA). Both accept a model, its tokenizer, and a list of sequences, and return the mean loss across the batch.

In [ ]:
def compute_mlm_loss(model, tokenizer, sequences, mask_prob=0.15, device='cuda', max_length=None):
    """
    Compute average masked-LM loss across a list of DNA sequences.

    For each sequence:
      1. Tokenize.
      2. Randomly mask `mask_prob` fraction of non-special-token positions.
      3. Forward pass through the model with masked positions labeled.
      4. Cross-entropy loss is computed only at masked positions (rest are -100).

    Returns the mean loss over all sequences in the batch.
    """
    # Put the model in evaluation mode (disables dropout, etc.)
    model.eval()
    model = model.to(device)
    losses = []

    # Disable gradient computation to save memory and speed up forward passes
    with torch.no_grad():
        for seq in sequences:
            # --- Tokenization ---
            # return_tensors='pt' gives PyTorch tensors
            # truncation=True limits length to the model's max context
            enc = tokenizer(
                seq,
                return_tensors='pt',
                truncation=True,
                max_length=max_length or tokenizer.model_max_length
            )
            input_ids = enc['input_ids'].to(device)

            # --- Identify which positions are eligible for masking ---
            # get_special_tokens_mask returns 1 for special tokens ([CLS], [SEP], etc.), 0 otherwise
            # We only mask real content tokens
            special_mask = torch.tensor(
                tokenizer.get_special_tokens_mask(input_ids[0].tolist(), already_has_special_tokens=True),
                dtype=torch.bool, device=device
            )
            mask_candidates = ~special_mask

            # --- Randomly choose 15% of candidate positions to mask ---
            rand = torch.rand(input_ids.shape[1], device=device)
            mask_positions = mask_candidates & (rand < mask_prob)

            # Skip this sequence if nothing was selected for masking (very short sequences)
            if mask_positions.sum() == 0:
                continue

            # --- Construct the masked input ---
            # Replace masked positions with the model's [MASK] token ID
            masked_input_ids = input_ids.clone()
            masked_input_ids[0, mask_positions] = tokenizer.mask_token_id

            # --- Build the loss mask ---
            # -100 is the standard PyTorch convention for "ignore this position in loss"
            # Only masked positions carry their true token as the target
            loss_labels = torch.full_like(input_ids, -100)
            loss_labels[0, mask_positions] = input_ids[0, mask_positions]

            # --- Forward pass ---
            # The model computes cross-entropy loss internally when `labels` is provided
            outputs = model(input_ids=masked_input_ids, labels=loss_labels)
            losses.append(outputs.loss.item())

    return float(np.mean(losses))


def compute_causal_loss(model, tokenizer, sequences, device='cuda', max_length=None):
    """
    Compute average next-token (causal) loss across a list of DNA sequences.

    Used for autoregressive models like HyenaDNA where MLM is not applicable.
    For each sequence, the model predicts each token from the tokens preceding it
    and returns cross-entropy loss averaged over all predicted positions.
    """
    model.eval()
    model = model.to(device)
    losses = []

    with torch.no_grad():
        for seq in sequences:
            # Tokenize with the same settings as MLM (though causal models handle sequence differently)
            enc = tokenizer(
                seq,
                return_tensors='pt',
                truncation=True,
                max_length=max_length or tokenizer.model_max_length
            )
            input_ids = enc['input_ids'].to(device)

            # For causal LM, passing labels=input_ids tells the model to predict each token
            # from the preceding ones. The model internally shifts the labels by one position.
            outputs = model(input_ids=input_ids, labels=input_ids)
            losses.append(outputs.loss.item())

    return float(np.mean(losses))

print('Helper functions defined.')

## Nucleotide Transformer v2 evaluation

First execution of this cell downloads the model weights (about 2 GB) from HuggingFace and caches them in the project's `models/` directory. Subsequent runs load from cache in a few seconds. GPU memory usage during inference: approximately 2 GB for weights plus 1 GB for activations per sequence.

After computing the loss, the model is explicitly deleted and CUDA cache is cleared to free VRAM before loading the next model.

In [ ]:
# HuggingFace model identifier for Nucleotide Transformer v2 (500M multi-species)
NT_MODEL_ID = 'InstaDeepAI/nucleotide-transformer-v2-500m-multi-species'

print(f'Loading tokenizer + model for {NT_MODEL_ID}...')
# trust_remote_code=True is required to load the custom modeling code shipped with this repo
nt_tokenizer = AutoTokenizer.from_pretrained(NT_MODEL_ID, trust_remote_code=True)
nt_model = AutoModelForMaskedLM.from_pretrained(NT_MODEL_ID, trust_remote_code=True)
print(f'Loaded. Vocab size: {nt_tokenizer.vocab_size:,}')

# --- Compute mean MLM loss across the batch of mosquito sequences ---
print('\nComputing MLM loss on 50 sequences...')
nt_loss = compute_mlm_loss(nt_model, nt_tokenizer, sequences, device=DEVICE)

# --- Report results ---
# The random-baseline reference (log of vocab size) is the loss a maximally uncertain model
# would produce. Actual loss should be well below this if the model has learned mosquito structure.
print(f'\nNT-v2 mean MLM loss:       {nt_loss:.4f}')
print(f'NT-v2 perplexity:          {np.exp(nt_loss):.2f}')
print(f'Random-baseline reference: {np.log(nt_tokenizer.vocab_size):.4f}  (worst case)')

# --- Free GPU memory before loading the next model ---
# del removes the Python reference; empty_cache asks CUDA to release the underlying VRAM
del nt_model
if DEVICE == 'cuda':
    torch.cuda.empty_cache()

## DNABERT-2 evaluation

DNABERT-2 is a smaller model (117M parameters) using byte-pair encoding rather than fixed k-mer tokenization. It shares the masked language modeling objective with NT-v2, so their losses are directly comparable.

The primary research question this cell contributes to is whether BPE tokenization confers an advantage in modeling *An. gambiae* sequence content relative to fixed 6-mer tokenization. A lower loss for DNABERT-2 than NT-v2 would support the hypothesis that biologically meaningful motif-level tokens (learned by BPE) help the model predict mosquito DNA.

**Loading notes.** DNABERT-2 uses the MosaicBERT architecture, not vanilla BERT, and ships custom config and model code via `trust_remote_code=True`. Standard `AutoModelForMaskedLM.from_pretrained()` fails on this model with a config-class identity mismatch because transformers' AutoModel registry enforces strict type checking. The workaround used in the code below bypasses `AutoModel` entirely, importing DNABERT-2's own `BertForMaskedLM` class directly from the cached remote code via `get_class_from_dynamic_module`. This gives a clean checkpoint load with all 117M parameters populated from the pretrained weights, rather than the mostly-random initialization that occurs when the wrong model class is used.

In [ ]:
# HuggingFace model identifier for DNABERT-2 (117M parameters)
DNABERT2_MODEL_ID = 'zhihan1996/DNABERT-2-117M'

# --- DNABERT-2 requires a custom loading path ---
# Standard AutoModelForMaskedLM.from_pretrained() fails on this model because DNABERT-2
# uses MosaicBERT architecture and ships a custom BertConfig class via trust_remote_code.
# transformers' internal AutoModel.register() enforces a strict type-identity check between
# the config's class and the model's config_class attribute, which fails when the config
# comes from transformers_modules but the model class is the built-in BertForMaskedLM.
#
# The workaround: bypass AutoModel entirely. Import DNABERT-2's OWN BertForMaskedLM class
# directly from its remote code (which knows how to consume the MosaicBERT-format checkpoint),
# then call from_pretrained on that class. Same trust_remote_code=True mechanism, just
# skipping the auto-registry layer.
from transformers import AutoConfig
from transformers.dynamic_module_utils import get_class_from_dynamic_module

print(f'Loading {DNABERT2_MODEL_ID}...')

# --- Tokenizer via standard trust_remote_code path ---
dnabert2_tokenizer = AutoTokenizer.from_pretrained(DNABERT2_MODEL_ID, trust_remote_code=True)

# --- Config via AutoConfig with trust_remote_code=True ---
# This produces the custom BertConfig class from transformers_modules, which DNABERT-2's
# own BertForMaskedLM expects.
dnabert2_config = AutoConfig.from_pretrained(DNABERT2_MODEL_ID, trust_remote_code=True)

# --- Import DNABERT-2's own BertForMaskedLM class directly from its remote code ---
# get_class_from_dynamic_module fetches the class from transformers_modules (already cached
# on disk from the trust_remote_code download). No AutoModel registry involvement.
BertForMaskedLM_dnabert2 = get_class_from_dynamic_module(
    'bert_layers.BertForMaskedLM',
    DNABERT2_MODEL_ID,
)

# --- Instantiate the model with custom class + custom config ---
# The checkpoint's parameter names now map cleanly to this class's module structure,
# so all 117M weights load from the checkpoint (previously most were randomly initialized).
dnabert2_model = BertForMaskedLM_dnabert2.from_pretrained(DNABERT2_MODEL_ID, config=dnabert2_config)
print(f'Loaded. Vocab size: {dnabert2_tokenizer.vocab_size:,}')
print(f'Total parameters: {sum(p.numel() for p in dnabert2_model.parameters()):,}')

# --- Compute mean MLM loss on the same batch of mosquito sequences ---
print('\nComputing MLM loss on 50 sequences...')
dnabert2_loss = compute_mlm_loss(dnabert2_model, dnabert2_tokenizer, sequences, device=DEVICE)

print(f'\nDNABERT-2 mean MLM loss:   {dnabert2_loss:.4f}')
print(f'DNABERT-2 perplexity:      {np.exp(dnabert2_loss):.2f}')
print(f'Random-baseline reference: {np.log(dnabert2_tokenizer.vocab_size):.4f}  (worst case)')

# --- Free VRAM for the next model ---
del dnabert2_model
if DEVICE == 'cuda':
    torch.cuda.empty_cache()

## HyenaDNA evaluation

HyenaDNA is autoregressive (causal LM), not masked LM. It predicts each token from the preceding context. The evaluation therefore uses `compute_causal_loss` rather than `compute_mlm_loss`.

Numerical caveat: HyenaDNA's loss and perplexity are not directly comparable to NT-v2 and DNABERT-2's masked LM values because next-token prediction over a 4-token single-nucleotide vocabulary is a fundamentally different task than filling a masked position from a 4,000-token vocabulary. HyenaDNA is still evaluated here for completeness, but the primary head-to-head comparison is NT-v2 versus DNABERT-2.

In [ ]:
# HuggingFace model identifier for HyenaDNA medium (450 kb max context)
HYENA_MODEL_ID = 'LongSafari/hyenadna-medium-450k-seqlen-hf'

print(f'Loading {HYENA_MODEL_ID}...')
hyena_tokenizer = AutoTokenizer.from_pretrained(HYENA_MODEL_ID, trust_remote_code=True)
hyena_model = AutoModelForCausalLM.from_pretrained(HYENA_MODEL_ID, trust_remote_code=True)
print(f'Loaded. Vocab size: {hyena_tokenizer.vocab_size:,}')

# --- Compute mean causal LM loss (different from MLM loss above) ---
print('\nComputing causal LM loss on 50 sequences...')
hyena_loss = compute_causal_loss(hyena_model, hyena_tokenizer, sequences, device=DEVICE)

print(f'\nHyenaDNA mean causal loss: {hyena_loss:.4f}')
print(f'HyenaDNA perplexity:       {np.exp(hyena_loss):.2f}')
print(f'Random-baseline reference: {np.log(hyena_tokenizer.vocab_size):.4f}  (worst case)')

# --- Free VRAM ---
del hyena_model
if DEVICE == 'cuda':
    torch.cuda.empty_cache()

## Results comparison

A summary table and bar chart of the three models' losses. NT-v2 and DNABERT-2 (both MLM) are colored blue to indicate direct comparability. HyenaDNA (causal) is colored orange to flag that its numeric loss is on a different scale.

The figure is saved to `outputs/perplexity_comparison.png` for use in reports and email attachments.

In [ ]:
# --- Build the summary table, skipping any model that failed to load ---
# Each entry is (label, loss, loss_type, vocab_size). None loss values indicate a failed load.
rows_raw = [
    ('NT-v2 (6-mer, MLM)',           nt_loss,       'MLM',    nt_tokenizer.vocab_size),
    ('DNABERT-2 (BPE, MLM)',         dnabert2_loss, 'MLM',    dnabert2_tokenizer.vocab_size if dnabert2_tokenizer else None),
    ('HyenaDNA (single-nt, causal)', hyena_loss,    'Causal', hyena_tokenizer.vocab_size),
]

# Filter out rows whose loss is None (model did not load)
rows = [r for r in rows_raw if r[1] is not None]

results = pd.DataFrame(rows, columns=['model', 'loss', 'loss_type', 'vocab_size'])
results['perplexity'] = np.exp(results['loss'])
print(results.to_string(index=False))

# --- Bar chart of mean loss per available model ---
fig, ax = plt.subplots(figsize=(9, 5))

# Color code: blue for MLM (comparable), orange for causal (different scale)
colors = ['tab:blue' if lt == 'MLM' else 'tab:orange' for lt in results['loss_type']]
bars = ax.bar(results['model'], results['loss'], color=colors)

ax.set_ylabel('Mean loss (cross-entropy per predicted token)')
ax.set_title('DNA foundation model perplexity comparison on An. gambiae 3L windows')
ax.tick_params(axis='x', rotation=15)

# Numeric labels on top of each bar
for bar, val in zip(bars, results['loss']):
    ax.text(bar.get_x() + bar.get_width() / 2, val, f'{val:.3f}', ha='center', va='bottom')

# Legend explaining the color coding
from matplotlib.patches import Patch
legend_patches = []
if any(lt == 'MLM' for lt in results['loss_type']):
    legend_patches.append(Patch(color='tab:blue', label='MLM loss (directly comparable)'))
if any(lt == 'Causal' for lt in results['loss_type']):
    legend_patches.append(Patch(color='tab:orange', label='Causal loss (different objective)'))
if legend_patches:
    ax.legend(handles=legend_patches)

plt.tight_layout()

# --- Save the figure to outputs/ for later reference ---
out_path = OUTPUTS_DIR / 'perplexity_comparison.png'
plt.savefig(out_path, dpi=120)
print(f'\nSaved figure to {out_path}')
plt.show()

# --- Notify if any models were skipped ---
skipped = [r[0] for r in rows_raw if r[1] is None]
if skipped:
    print(f'\nModels skipped in this run: {skipped}')

## Interpretation

Three questions the results address:

**1. Between NT-v2 and DNABERT-2, which has lower MLM loss?**

This is the primary head-to-head comparison. Lower DNABERT-2 loss would support the hypothesis that BPE tokenization captures biologically meaningful motifs in a way that aids prediction. Lower NT-v2 loss would suggest fixed 6-mer positional predictability outweighs the flexibility advantage of BPE for this data.

**2. How does HyenaDNA compare?**

Its causal loss will typically be lower in absolute terms because next-token prediction over 4 tokens is easier than mask-filling over thousands. To compare fairly across objectives, normalized perplexity (loss / log(vocab_size)) or per-base-pair loss is more informative than raw loss. A future iteration can add these.

**3. How close is each model to its random-baseline loss?**

The gap between measured loss and `log(vocab_size)` indicates how much structure the model has learned relative to a maximally uncertain baseline. Losses near the baseline suggest the model has not internalized *An. gambiae* patterns during pretraining. Losses well below the baseline (e.g., 30 percent of baseline) indicate substantial learned structure.

**Current-run caveats:**

- 50 sequences is a small sample. For a publishable comparison, running on 1000+ sequences and reporting bootstrap confidence intervals is standard.
- Only one mask draw per sequence. Randomness in mask positions adds noise. Averaging multiple mask draws per sequence would tighten the estimates.
- Only chromosome 3L is sampled. Cross-chromosome comparison could reveal whether some parts of the genome are easier to predict than others.

**Next step:** notebook 03 uses this same framework to compare pg-gan-mosquito-simulated sequences against real AgamP4 sequences, using DNA-LM perplexity as an independent judge of simulator quality.